In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc, f1_score, confusion_matrix

# Global result logging structure
comparison_records = []

def train_and_evaluate(dataset_name, X_tr, y_tr, X_te, y_te):
    # --- 1. Baseline Model (Logistic Regression) ---
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    
    preds_lr = lr.predict(X_te)
    probs_lr = lr.predict_proba(X_te)[:, 1]
    
    p_lr, r_lr, _ = precision_recall_curve(y_te, probs_lr)
    auc_pr_lr = auc(r_lr, p_lr)
    f1_lr = f1_score(y_te, preds_lr)
    
    # Save baseline artifact
    joblib.dump(lr, f'../models/{dataset_name}_logistic_regression.pkl')
    
    # --- 2. Tuned Ensemble Model (Random Forest) ---
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    
    preds_rf = rf.predict(X_te)
    probs_rf = rf.predict_proba(X_te)[:, 1]
    
    p_rf, r_rf, _ = precision_recall_curve(y_te, probs_rf)
    auc_pr_rf = auc(r_rf, p_rf)
    f1_rf = f1_score(y_te, preds_rf)
    
    # Save ensemble artifact
    joblib.dump(rf, f'../models/{dataset_name}_random_forest.pkl')
    
    # Log Comparison Results
    comparison_records.append({
        'Dataset': dataset_name, 'Model': 'Logistic Regression', 'F1-Score': f1_lr, 'AUC-PR': auc_pr_lr
    })
    comparison_records.append({
        'Dataset': dataset_name, 'Model': 'Random Forest', 'F1-Score': f1_rf, 'AUC-PR': auc_pr_rf
    })
    
    print(f"\n=== Evaluation: {dataset_name} ===")
    print(f"LR  - F1: {f1_lr:.4f} | AUC-PR: {auc_pr_lr:.4f}")
    print(f"RF  - F1: {f1_rf:.4f} | AUC-PR: {auc_pr_rf:.4f}")

# Execute for both datasets
train_and_evaluate("Fraud_Data", 
                   np.load('../data/processed/X_train_f.npy'), np.load('../data/processed/y_train_f.npy'),
                   np.load('../data/processed/X_test_f.npy'), np.load('../data/processed/y_test_f.npy'))

train_and_evaluate("CreditCard", 
                   np.load('../data/processed/X_train_c.npy'), np.load('../data/processed/y_train_c.npy'),
                   np.load('../data/processed/X_test_c.npy'), np.load('../data/processed/y_test_c.npy'))

# Display Metrics Table (Rubric 2.4)
df_comparison = pd.DataFrame(comparison_records)
print("\n--- MODEL COMPARISON SUMMARY ---")
print(df_comparison.to_string(index=False))

In [ ]:
# Task 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import joblib

# Dictionary to hold final model performance metrics for side-by-side comparison
model_results = {}

def evaluate_model(y_true, y_pred, y_prob, model_name):
    """
    Computes standard metrics for imbalanced datasets: F1-Score, Confusion Matrix, and AUC-PR.
    """
    # Calculate Precision-Recall Curve & AUC-PR
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    auc_pr = auc(recall, precision)
    f1 = f1_score(y_true, y_pred)
    
    print(f"=== {model_name} Evaluation ===")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC-PR: {auc_pr:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    plt.title(f'{model_name} Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    
    # Save metrics for cross-model comparison
    return {"F1-Score": f1, "AUC-PR": auc_pr}

In [ ]:
# Assuming you saved these from Task 1
X_train = np.load('../data/processed/X_train_resampled.npy')
y_train = np.load('../data/processed/y_train_resampled.npy')
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

In [ ]:
# Initialize and fit Baseline Model
lr_baseline = LogisticRegression(max_iter=1000, random_state=42)
lr_baseline.fit(X_train, y_train)

# Predict on validation/test set
y_pred_lr = lr_baseline.predict(X_test)
y_prob_lr = lr_baseline.predict_proba(X_test)[:, 1]

# Evaluate
model_results["Logistic Regression"] = evaluate_model(y_test, y_pred_lr, y_prob_lr, "Logistic Regression Baseline")

# Save model artifact
joblib.dump(lr_baseline, '../models/logistic_regression_baseline.pkl')

In [ ]:
# Initialize Random Forest with basic hyperparameters
rf_ensemble = RandomForestClassifier(
    n_estimators=150, 
    max_depth=12, 
    random_state=42, 
    n_jobs=-1
)
rf_ensemble.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_ensemble.predict(X_test)
y_prob_rf = rf_ensemble.predict_proba(X_test)[:, 1]

# Evaluate
model_results["Random Forest"] = evaluate_model(y_test, y_pred_rf, y_prob_rf, "Random Forest Ensemble")

# Save model artifact
joblib.dump(rf_ensemble, '../models/random_forest_ensemble.pkl')

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1_scores = []
cv_auc_pr_scores = []

print("Starting 5-Fold Stratified Cross-Validation on Random Forest Ensemble...")

# Iterate through folds
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    
    # Clone and fit model for the current fold
    fold_model = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
    fold_model.fit(X_fold_train, y_fold_train)
    
    # Evaluate on fold validation data
    preds = fold_model.predict(X_fold_val)
    probs = fold_model.predict_proba(X_fold_val)[:, 1]
    
    # Calculate PR curve metric
    precision, recall, _ = precision_recall_curve(y_fold_val, probs)
    cv_auc_pr_scores.append(auc(recall, precision))
    cv_f1_scores.append(f1_score(y_fold_val, preds))
    
    print(f"Fold {fold} - F1-Score: {cv_f1_scores[-1]:.4f} | AUC-PR: {cv_auc_pr_scores[-1]:.4f}")

# Report mean and standard deviation
print(f"\n=== CV Results Summary ===")
print(f"Mean F1-Score: {np.mean(cv_f1_scores):.4f} (±{np.std(cv_f1_scores):.4f})")
print(f"Mean AUC-PR: {np.mean(cv_auc_pr_scores):.4f} (±{np.std(cv_auc_pr_scores):.4f})")

In [ ]:
df_compare = pd.DataFrame(model_results).T
print("\n=== Model Evaluation Summary Table ===")
print(df_compare)

# Plot performance side-by-side
df_compare.plot(kind='bar', figsize=(8, 5))
plt.title('Model Comparison: Baseline vs. Ensemble')
plt.ylabel('Score Value')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, 1.1)
plt.legend(loc='lower left')
plt.show()